In [2]:
import pandas as pd
import gzip
import json
from pm4py.utils import format_dataframe
from pm4py import write_xes
from tqdm.notebook import tqdm
from copy import deepcopy

In [5]:
dataset_json_path = r".out/eventlogs/xbpic15-0.3-1.json.gz"
new_dataset_json_path = r".out/eventlogs/bpic15-0.3-1.json.gz"
with gzip.open(dataset_json_path, "r") as f:
        data = f.read()
        j = json.loads(data.decode('utf-8'))
j_sampled = deepcopy(j)


In [ ]:
print(j["cases"][3]["attributes"]) #{'IDofConceptCase': '10083477', 'Responsible_actor': '560912', 'SUMleges': '339.4761', 'caseStatus': 'O', 'case_type': '557669', 'concept:name': '10083315', 'label': 'normal', 'last_phase': 'Beschikking verzonden', 'parts': 'Bouw', 'requestComplete': 'FALSE', 'startDate': '2014-04-17T00:00:00+02:00', 'termName': 'Termijn bezwaar en beroep 1'}
print(j["cases"][3]["events"][0]) #{'attributes': {'action_code': '01_HOOFD_010', 'activityNameEN': 'register submission date request', 'activityNameNL': 'registratie datum binnenkomst aanvraag', 'concept:name': '01_HOOFD_010', 'dateFinished': '2014-04-25 00:00:00', 'dueDate': '2014-04-28T10:46:46+02:00', 'lifecycle:transition': 'complete', 'monitoringResource': '560912', 'org:resource': '560872', 'planned': '2014-04-24T10:46:46+02:00', 'question': 'EMPTY', 'time:timestamp': '2014-04-17T00:00:00+02:00'}, 'name': '01_HOOFD_010+complete', 'timestamp': '2014-04-17T00:00:00+02:00', 'timestamp_end': None}


"""
{'AMOUNT_REQ': '20000',
'REG_DATE': '2011-10-01T00:38:44.546+02:00',
'concept:name': '173688',
'label': 'normal'}

{'concept:name': 'Accepted', 
'impact': 'Low', 
'lifecycle:transition': 'In Progress', 
'org:group': 'Org line A2',
'org:resource': 'Tomas', 
'org:role': 'A2_2', 
'organization country': 'cn', 
'organization involved': 'M1 2nd', 
'product': 'PROD753', 
'resource country': 'Sweden',
'time:timestamp': '2007-05-10T16:21:54+02:00'}, 
'name': 'Accepted+In Progress',
'timestamp': '2007-05-10T16:21:54+02:00', 
'timestamp_end': None}

{'action_code': '01_HOOFD_010',
'activityNameEN': 'register submission date request',
'activityNameNL': 'registratie datum binnenkomst aanvraag',
'concept:name': '01_HOOFD_010', 
'dateFinished': '2014-04-25 00:00:00',
'dueDate': '2014-04-28T10:46:46+02:00',
'lifecycle:transition': 'complete',
'monitoringResource': '560912',
'org:resource': '560872',
'planned': '2014-04-24T10:46:46+02:00',
'question': 'EMPTY',
'time:timestamp': '2014-04-17T00:00:00+02:00'},
'name': '01_HOOFD_010+complete', 
'timestamp': '2014-04-17T00:00:00+02:00',
'timestamp_end': None}


"""
# check how many of the cases are normal and how many are anomlay
normal = 0
anomaly = 0
for case in j_sampled["cases"]:
    if case["attributes"]["label"] == "normal":
        normal += 1
    else:
        anomaly += 1
print(f"normal: {normal}, anomaly: {anomaly}")

{'IDofConceptCase': '10083477', 'Responsible_actor': '560912', 'SUMleges': '339.4761', 'caseStatus': 'O', 'case_type': '557669', 'concept:name': '10083315', 'label': 'normal', 'last_phase': 'Beschikking verzonden', 'parts': 'Bouw', 'requestComplete': 'FALSE', 'startDate': '2014-04-17T00:00:00+02:00', 'termName': 'Termijn bezwaar en beroep 1'}
{'attributes': {'action_code': '01_HOOFD_010', 'activityNameEN': 'register submission date request', 'activityNameNL': 'registratie datum binnenkomst aanvraag', 'concept:name': '01_HOOFD_010', 'dateFinished': '2014-04-25 00:00:00', 'dueDate': '2014-04-28T10:46:46+02:00', 'lifecycle:transition': 'complete', 'monitoringResource': '560912', 'org:resource': '560872', 'planned': '2014-04-24T10:46:46+02:00', 'question': 'EMPTY', 'time:timestamp': '2014-04-17T00:00:00+02:00'}, 'name': '01_HOOFD_010+complete', 'timestamp': '2014-04-17T00:00:00+02:00', 'timestamp_end': None}
normal: 833, anomaly: 366


In [9]:
print(j["cases"][3]["attributes"]) #{'IDofConceptCase': '10083477', 'Responsible_actor': '560912', 'SUMleges': '339.4761', 'caseStatus': 'O', 'case_type': '557669', 'concept:name': '10083315', 'label': 'normal', 'last_phase': 'Beschikking verzonden', 'parts': 'Bouw', 'requestComplete': 'FALSE', 'startDate': '2014-04-17T00:00:00+02:00', 'termName': 'Termijn bezwaar en beroep 1'}
print(j["cases"][3]["events"][0]) #{'attributes': {'action_code': '01_HOOFD_010', 'activityNameEN': 'register submission date request', 'activityNameNL': 'registratie datum binnenkomst aanvraag', 'concept:name': '01_HOOFD_010', 'dateFinished': '2014-04-25 00:00:00', 'dueDate': '2014-04-28T10:46:46+02:00', 'lifecycle:transition': 'complete', 'monitoringResource': '560912', 'org:resource': '560872', 'planned': '2014-04-24T10:46:46+02:00', 'question': 'EMPTY', 'time:timestamp': '2014-04-17T00:00:00+02:00'}, 'name': '01_HOOFD_010+complete', 'timestamp': '2014-04-17T00:00:00+02:00', 'timestamp_end': None}

def remove_specific_keys_in_event_attributes(event):
    keys_to_remove = ['action_code', 'activityNameEN', 'activityNameNL', 'dateFinished', 'dueDate', 'lifecycle:transition', 'monitoringResource', 'planned', 'question']
    for key in keys_to_remove:
        event["attributes"].pop(key, None)
    return event

def remove_specific_keys_in_case_attributes(case):
    keys_to_remove = ['IDofConceptCase', 'Responsible_actor', 'SUMleges', 'caseStatus', 'case_type', 'last_phase', 'parts', 'requestComplete', 'startDate', 'termName']
    for key in keys_to_remove:
        case["attributes"].pop(key, None)
    return case

def remove_keys_in_case_events(case):
    case["events"] = [remove_specific_keys_in_event_attributes(event) for event in case["events"]]
    return case

def modify_case(case):
    case = remove_specific_keys_in_case_attributes(case)
    case = remove_keys_in_case_events(case)
    return case

def modify_event_log_j(j_orig):
    j_modified = deepcopy(j_orig)
    j_modified["cases"] = [modify_case(case) for case in j_modified["cases"]]
    return j_modified

j_new = modify_event_log_j(j_sampled)

print(j_new["cases"][3]["attributes"])
print(j_new["cases"][3]["events"][0])


{'IDofConceptCase': '10083477', 'Responsible_actor': '560912', 'SUMleges': '339.4761', 'caseStatus': 'O', 'case_type': '557669', 'concept:name': '10083315', 'label': 'normal', 'last_phase': 'Beschikking verzonden', 'parts': 'Bouw', 'requestComplete': 'FALSE', 'startDate': '2014-04-17T00:00:00+02:00', 'termName': 'Termijn bezwaar en beroep 1'}
{'attributes': {'action_code': '01_HOOFD_010', 'activityNameEN': 'register submission date request', 'activityNameNL': 'registratie datum binnenkomst aanvraag', 'concept:name': '01_HOOFD_010', 'dateFinished': '2014-04-25 00:00:00', 'dueDate': '2014-04-28T10:46:46+02:00', 'lifecycle:transition': 'complete', 'monitoringResource': '560912', 'org:resource': '560872', 'planned': '2014-04-24T10:46:46+02:00', 'question': 'EMPTY', 'time:timestamp': '2014-04-17T00:00:00+02:00'}, 'name': '01_HOOFD_010+complete', 'timestamp': '2014-04-17T00:00:00+02:00', 'timestamp_end': None}
{'concept:name': '10083315', 'label': 'normal'}
{'attributes': {'concept:name': '0

In [10]:
import numpy as np
j = j_new
# Get the total number of cases
orig_num_cases = len(j["cases"])
print(f"Total number of orig cases: {orig_num_cases}")

# Sample 6000 cases or all cases if less than 6000
# sample_size = min(6000, num_cases)
# sampled_indices = np.random.choice(num_cases, sample_size, replace=False)

# Create j_sampled with the sampled cases
j_sampled = deepcopy(j)

# select only cases whole length is less than 15
j_sampled["cases"] = [j["cases"][i] for i in range(orig_num_cases) if len(j["cases"][i]["events"]) < 50]

new_num_cases = len(j_sampled["cases"])
print(f"Number of cases after filtering: {new_num_cases}")
sample_size = min(6000, new_num_cases)
sampled_indices = np.random.choice(new_num_cases, sample_size, replace=False)

j_sampled["cases"] = [j_sampled["cases"][i] for i in sampled_indices]

print(f"Number of final sampled cases: {len(j_sampled['cases'])}")
normal = 0
anomaly = 0
for case in j_sampled["cases"]:
    if case["attributes"]["label"] == "normal":
        normal += 1
    else:
        anomaly += 1
print(f"normal: {normal}, anomaly: {anomaly}")


Total number of orig cases: 1199
Number of cases after filtering: 803
Number of final sampled cases: 803
normal: 562, anomaly: 241


In [11]:

# Save the sampled data to a new file
with gzip.open(new_dataset_json_path, "w") as f:
    f.write(json.dumps(j_sampled).encode('utf-8'))


In [12]:
def binet_to_df(path):
    with gzip.open(path, "r") as f:
        data = f.read()
        j = json.loads(data.decode('utf-8'))
    
    res_list = []
    
    for case in j['cases']:
        trace = pd.DataFrame.from_dict(case['events'])
        trace['anomaly'] = case['attributes']['label'] if isinstance(case['attributes']['label'], str) else case['attributes']['label']['anomaly']
        trace['trace_id'] = case['id']
        res_list.append(trace)
    
    if res_list:
        res = pd.concat(res_list, ignore_index=True)
        res = pd.concat([res.drop(['attributes'], axis=1), res['attributes'].apply(pd.Series)], axis=1)
    else:
        res = pd.DataFrame()
    
    return res

from datetime import datetime, timedelta

def assign_sequential_timestamps(df, start_time=None, step_minutes=10, duration_minutes=5):
    if start_time is None:
        start_time = datetime.now()

    df = df.copy()
    timestamps = []
    timestamps_end = []

    for trace_id, group in df.groupby('trace_id'):
        base_time = start_time
        for _ in range(len(group)):
            timestamps.append(base_time)
            timestamps_end.append(base_time + timedelta(minutes=duration_minutes))
            base_time += timedelta(minutes=step_minutes)

    df['timestamp'] = timestamps
    df['timestamp_end'] = timestamps_end
    return df

def convert_to_pm4py_df(df):
    df = df.copy()
    df = df.drop(["concept:name"], axis=1)
    df = df.rename(columns={'name': 'activity', 'trace_id':'case_id', 'user':'org:resource', 'anomaly': 'anomaly'})
    df = df.astype({'activity': str, 'anomaly': str, 'org:resource': str})
    df = format_dataframe(df, case_id='case_id',activity_key='activity', timestamp_key='timestamp')
    df = df.drop(['activity', 'timestamp', 'timestamp_end'], axis=1)
    return df

def convert_and_write_json_to_xes(path_to_json, path_to_xes):
    df = binet_to_df(path_to_json)
    df = assign_sequential_timestamps(df)
    df = convert_to_pm4py_df(df)
    write_xes(df, path_to_xes)

In [13]:
# dataset_names = ["medium", "small", "p2p", "paper"]
dataset_names = ["bpic15-0.3"]
dataset_json_path = r".out/eventlogs/bpic15-0.3-1.json.gz"
dataset_xes_path = r".out/eventlogs/bpic15-0.3-1.xes"
# convert_and_write_json_to_xes(dataset_json_path, dataset_xes_path)

In [14]:
mydf = binet_to_df(dataset_json_path)

In [15]:
mydf.shape

(28215, 9)

In [16]:
convertdf = convert_to_pm4py_df(mydf)

/home/devashish/anaconda3/envs/ltn/lib/python3.9/site-packages/pm4py/utils.py:132: UserWarning: Some rows of the Pandas data frame have been removed because of empty case IDs, activity labels, or timestamps to ensure the correct functioning of PM4Py's algorithms.
  warnings.warn(
/home/devashish/anaconda3/envs/ltn/lib/python3.9/site-packages/pm4py/utils.py:137: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[constants.CASE_CONCEPT_NAME] = df[constants.CASE_CONCEPT_NAME].astype(
/home/devashish/anaconda3/envs/ltn/lib/python3.9/site-packages/pm4py/utils.py:141: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: ht

In [17]:
convertdf.head(1)

,anomaly,case_id,org:resource,dateStop,case:concept:name,concept:name,time:timestamp,@@index,@@case_index
0,normal,10009138,9264148,NaT,10009138,01_HOOFD_010+complete,2014-04-10 22:00:00+00:00,0,0


In [18]:
write_xes(convertdf, dataset_xes_path)  

exporting log, completed traces ::   0%|          | 0/803 [00:00<?, ?it/s]